In [20]:
import pandas as pd
import requests
import io
import numpy as np

In [21]:
url = "https://earthquake.usgs.gov/fdsnws/event/1/query"
params = {
    "format": "geojson",
    "limit": 1
}

response = requests.get(url, params=params)
data = response.json()

print(data)

{'type': 'FeatureCollection', 'metadata': {'generated': 1795106570000, 'url': 'https://earthquake.usgs.gov/fdsnws/event/1/query?format=geojson&limit=1', 'title': 'USGS Earthquakes', 'status': 200, 'api': '2.8.4', 'limit': 1, 'offset': 1}, 'features': [{'type': 'Feature', 'properties': {'mag': 0.7, 'place': '84 km NW of Karluk, Alaska', 'time': 1795105268366, 'updated': 1795105433993, 'tz': None, 'url': 'https://earthquake.usgs.gov/earthquakes/eventpage/aka2026wxztsg', 'detail': 'https://earthquake.usgs.gov/fdsnws/event/1/query?eventid=aka2026wxztsg&format=geojson', 'felt': None, 'cdi': None, 'mmi': None, 'alert': None, 'status': 'automatic', 'tsunami': 0, 'sig': 8, 'net': 'ak', 'code': 'a2026wxztsg', 'ids': ',aka2026wxztsg,', 'sources': ',ak,', 'types': ',origin,phase-data,', 'nst': 6, 'dmin': 0, 'rms': 0.1, 'gap': 111, 'magType': 'ml', 'type': 'earthquake', 'title': 'M 0.7 - 84 km NW of Karluk, Alaska'}, 'geometry': {'type': 'Point', 'coordinates': [-155.324, 58.174, 5]}, 'id': 'aka20

In [22]:
response

<Response [200]>

In [23]:
df = pd.json_normalize(data["features"])

N = len(df)
print(f"Target population (N): {N}")
df.head()

Target population (N): 1


,type,id,properties.mag,properties.place,properties.time,properties.updated,properties.tz,properties.url,properties.detail,properties.felt,...,properties.types,properties.nst,properties.dmin,properties.rms,properties.gap,properties.magType,properties.type,properties.title,geometry.type,geometry.coordinates
0,Feature,aka2026wxztsg,0.7,"84 km NW of Karluk, Alaska",1795105268366,1795105433993,None,https://earthquake.usgs.gov/earthquakes/eventp...,https://earthquake.usgs.gov/fdsnws/event/1/que...,None,...,",origin,phase-data,",6,0,0.1,111,ml,earthquake,"M 0.7 - 84 km NW of Karluk, Alaska",Point,"[-155.324, 58.174, 5]"


In [25]:
url = "https://earthquake.usgs.gov/fdsnws/event/1/query"
params = {
    "format": "csv",
    "eventtype": "earthquake",
    "minmagnitude": 2.0,
    "starttime": "2026-10-01",
    "endtime": "2026-10-31",
    "limit": 20000
}

response = requests.get(url, params=params)
df = pd.read_csv(io.StringIO(response.text))

N = len(df)
print(f"Target population (N): {N}")
df.head()

Target population (N): 3744


,time,latitude,longitude,depth,mag,magType,nst,gap,dmin,rms,...,updated,place,type,horizontalError,depthError,magError,magNst,status,locationSource,magSource
0,2026-10-30T23:56:32.771Z,-14.8682,168.1797,10.000,4.5,mb,21,83,6.760,0.42,...,2026-11-15T22:41:26.040Z,"120 km E of Port-Olry, Vanuatu",earthquake,11.49,1.913,0.140,15.0,reviewed,us,us
1,2026-10-30T23:54:14.074Z,-30.2201,-177.9668,10.000,4.4,mb,11,178,0.972,0.21,...,2026-11-16T01:04:46.040Z,"Kermadec Islands, New Zealand",earthquake,12.52,1.948,0.178,9.0,reviewed,us,us
2,2026-10-30T23:06:11.549Z,62.3670,-154.9180,5.000,2.9,ml,37,80,1.000,0.60,...,2026-11-17T04:41:01.040Z,"74 km SSE of McGrath, Alaska",earthquake,3.20,0.000,0.100,26.0,reviewed,ak,ak
3,2026-10-30T22:58:32.275Z,-19.7554,-177.7678,518.336,4.2,mb,14,119,4.246,0.78,...,2026-11-15T04:40:52.040Z,Fiji region,earthquake,21.30,15.366,0.175,9.0,reviewed,us,us
4,2026-10-30T22:53:01.297Z,59.8890,-152.7850,102.500,3.1,ml,123,57,0.100,0.60,...,2026-11-17T04:34:06.040Z,"54 km WNW of Anchor Point, Alaska",earthquake,2.30,1.297,0.300,89.0,reviewed,ak,ak


In [26]:
df["time"] = pd.to_datetime(df["time"])

In [27]:
convenience_df = df.head(100).copy()

In [28]:
srs_df = df.sample(n=100, random_state=42).copy()

In [29]:
summary = pd.DataFrame({
    "Dataset": ["True Population (mu)", "Convenience Sample (x-bar)", "Simple Random Sample (x-bar)"],
    "Mean Magnitude": [
        df["mag"].mean(),
        convenience_df["mag"].mean(),
        srs_df["mag"].mean()
    ],
    "Start Time": [
        df["time"].min(),
        convenience_df["time"].min(),
        srs_df["time"].min()
    ],
    "End Time": [
        df["time"].max(),
        convenience_df["time"].max(),
        srs_df["time"].max()
    ],
    "Time Span": [
        df["time"].max() - df["time"].min(),
        convenience_df["time"].max() - convenience_df["time"].min(),
        srs_df["time"].max() - srs_df["time"].min()
    ]
})

print(summary.to_string(index=False))

                     Dataset  Mean Magnitude                       Start Time                         End Time               Time Span
        True Population (mu)        3.353133 2026-10-01 00:00:21.945000+00:00 2026-10-30 23:56:32.771000+00:00 29 days 23:56:10.826000
  Convenience Sample (x-bar)        3.023900 2026-10-30 01:05:02.221000+00:00 2026-10-30 23:56:32.771000+00:00  0 days 22:51:30.550000
Simple Random Sample (x-bar)        3.352900 2026-10-01 00:36:50.789000+00:00 2026-10-30 21:01:33.158000+00:00 29 days 20:24:42.369000


### Why does the convenience sample distort your estimate?

The sample is distorting my estimate because the USGS API automatically returns events in reverse chronological order. Also, all 100 of the Earthquakes occur in a short time span and the sample is influenced by any activity that occured right at the end of the month. These events do not represent the entire global distribution of earthquakes. This means that the sample mean will not be the same as the actual population mean. 

In [30]:
place_col = "properties.place" if "properties.place" in df.columns else "place"
mag_col = "properties.mag" if "properties.mag" in df.columns else "mag"

df["region"] = df[place_col].fillna("Unknown").apply(
    lambda x: x.split(",")[-1].strip() if "," in x else x.strip()
)

In [31]:
us_states_and_na = {
    "CA", "AK", "HI", "NV", "WA", "OR", "ID", "MT", "WY", "UT", "AZ", "NM", "CO", 
    "TX", "OK", "KS", "NE", "SD", "ND", "MN", "IA", "MO", "AR", "LA", "MS", "AL", 
    "TN", "KY", "IL", "IN", "OH", "MI", "WI", "FL", "GA", "SC", "NC", "VA", "WV", 
    "MD", "DE", "PA", "NJ", "NY", "CT", "RI", "MA", "VT", "NH", "ME", "Puerto Rico",
    "California", "Alaska", "Hawaii", "Nevada", "Washington", "Oregon", "Texas",
    "Mexico", "Canada", "Guatemala", "El Salvador", "Nicaragua", "Costa Rica", "Panama",
    "Dominican Republic", "U.S. Virgin Islands", "British Virgin Islands"
}

In [32]:
asia_pacific = {
    "Japan", "Indonesia", "Philippines", "Papua New Guinea", "Tonga", "Fiji", 
    "New Zealand", "Vanuatu", "Solomon Islands", "Taiwan", "China", "Russia", 
    "India", "Nepal", "Afghanistan", "Tajikistan", "Kyrgyzstan", "Uzbekistan",
    "Timor-Leste", "Mariana Islands", "Guam", "New Caledonia", "Kermadec Islands"
}

In [33]:
europe_africa = {
    "Greece", "Italy", "Turkey", "Türkiye", "Iceland", "Spain", "Portugal", 
    "France", "Romania", "Albania", "Croatia", "Norway", "United Kingdom",
    "Morocco", "Algeria", "Ethiopia", "Kenya", "Tanzania", "South Africa", "Iran", "Iraq"
}

In [34]:
def assign_stratum(region):
    if region in us_states_and_na:
        return "North America"
    elif region in asia_pacific:
        return "Asia-Pacific"
    elif region in europe_africa:
        return "Europe & Africa"
    else:
        return "South America & Oceanic"

df["stratum"] = df["region"].apply(assign_stratum)

In [35]:
print("Population Stratum Proportions:")
print(df["stratum"].value_counts(normalize=True).round(4))
print("-" * 45)

Population Stratum Proportions:
North America              0.5825
Asia-Pacific               0.2364
South America & Oceanic    0.1635
Europe & Africa            0.0176
Name: stratum, dtype: float64
---------------------------------------------


In [36]:
n_total = 200
strata_counts = df["stratum"].value_counts()
strata_props = strata_counts / len(df)

In [37]:
raw_sizes = strata_props * n_total
sample_sizes = np.floor(raw_sizes).astype(int)
remainder = n_total - sample_sizes.sum()

In [38]:
largest_remainders = (raw_sizes - sample_sizes).sort_values(ascending=False).index[:remainder]
sample_sizes[largest_remainders] += 1

In [39]:
stratified_df = pd.concat([
    df[df["stratum"] == stratum].sample(n=count, random_state=42)
    for stratum, count in sample_sizes.items()
])

In [40]:
print(f"Total Stratified Sample Size: {len(stratified_df)}")
print(stratified_df["stratum"].value_counts())
print("-" * 45)

Total Stratified Sample Size: 200
North America              116
Asia-Pacific                47
South America & Oceanic     33
Europe & Africa              4
Name: stratum, dtype: int64
---------------------------------------------


In [41]:
srs_200_df = df.sample(n=200, random_state=42)

In [42]:
srs_var_mean = srs_200_df[mag_col].var(ddof=1) / len(srs_200_df)

In [43]:
strat_var_mean = 0
for stratum, count in sample_sizes.items():
    W_h = strata_props[stratum]
    s_h_sq = stratified_df[stratified_df["stratum"] == stratum][mag_col].var(ddof=1)
    strat_var_mean += (W_h ** 2) * (s_h_sq / count)

In [44]:
print(f"True Population Mean (mu):               {df[mag_col].mean():.4f}")
print(f"Stratified Sample Mean (x-bar):          {stratified_df[mag_col].mean():.4f}")
print(f"SRS (n=200) Sample Mean (x-bar):         {srs_200_df[mag_col].mean():.4f}")
print(f"Stratified Variance of Sample Mean:      {strat_var_mean:.6f}")
print(f"SRS Variance of Sample Mean:             {srs_var_mean:.6f}")

True Population Mean (mu):               3.3531
Stratified Sample Mean (x-bar):          3.3249
SRS (n=200) Sample Mean (x-bar):         3.3366
Stratified Variance of Sample Mean:      0.002427
SRS Variance of Sample Mean:             0.006359


### Stratified Sample Mean vs. Simple Random Sample Mean

In a simple random sample, a random chance can cause one region to be overlooked or under-looked in any data set. For example, California and Alaska, have networks that can detect many small earthquakes compared to ocean area with larger earthquakes. Since earthquake magnitudes can have varied results across other regions, stratifying by geographic region would produce a more precise estimate of the true population mean magnitude. 